# 🔄 Część 4: nginx i Reverse Proxy

**Cel:** Zrozumieć czym jest nginx, czym różni się od Uvicorn/Gunicorn i po co go używamy.

**Analogia przewodnia:** 🍽️ Recepcjonista w restauracji

---

## 1. Czym jest nginx?

### nginx ma 3 role:

1. **HTTP Server** - serwuje statyczne pliki (HTML, CSS, JS, images)
2. **Reverse Proxy** - przekazuje requesty do innych serwerów (Uvicorn/Gunicorn)
3. **Load Balancer** - rozdziela ruch między wiele serwerów

**Najczęściej używamy nginx jako REVERSE PROXY.**

---

### Czym nginx NIE JEST:

❌ nginx **NIE URUCHAMIA** kodu Python
❌ nginx **NIE JEST** zamiennikiem dla Uvicorn/Gunicorn
❌ nginx **NIE IMPLEMENTUJE** WSGI/ASGI

**nginx to inna warstwa niż Uvicorn/Gunicorn!**

---

## 2. Reverse Proxy - co to dokładnie?

### Definicja:

**Reverse Proxy** = serwer stojący **PRZED** Twoją aplikacją, który:
- Odbiera requesty od klientów
- Przekazuje je do aplikacji backend (Uvicorn/Gunicorn)
- Odbiera odpowiedzi z aplikacji
- Wysyła je do klientów

### Schemat:

```
Klient (Browser)
    ↓ HTTP/HTTPS request
nginx (Reverse Proxy) - port 80/443
    ↓ HTTP request (internal)
Uvicorn/Gunicorn - port 8000
    ↓ ASGI/WSGI
FastAPI/Django (Python app)
```

**Klient NIE komunikuje się bezpośrednio z Uvicorn!**

Klient widzi tylko nginx (port 80/443), a nginx wewnętrznie przekazuje do Uvicorn (port 8000).

---

### 🍽️ Analogia: Recepcjonista w restauracji

**BEZ nginx (bezpośredni dostęp do Uvicorn):**

```
Klient → wchodzi prosto do kuchni → krzyczy zamówienie do kucharza
```

**Problemy:**
- Kucharz musi obsłużyć klienta + gotować (2 role!)
- Klient widzi kuchnię (brak prywatności)
- Chaos!

---

**Z nginx (reverse proxy):**

```
Klient → Recepcjonista (nginx) → Kuchnia (Uvicorn)
```

**Recepcjonista (nginx):**
1. Wita klienta
2. Przyjmuje zamówienie
3. Przekazuje do kuchni (Uvicorn)
4. Odbiera gotowe danie
5. Podaje klientowi

**Kucharz (Uvicorn):**
- Skupia się tylko na gotowaniu (wykonywaniu kodu Python)
- Nie musi obsługiwać klientów
- Nie jest widoczny z zewnątrz

**To jest reverse proxy!**

---

## 3. Po co nginx? (8 powodów)

### 1. Statyczne pliki

```
Request: GET /static/logo.png
    ↓
nginx: "To statyczny plik, obsłużę SAM" (SZYBKO!)
    ↓
Zwraca plik z dysku
    ↓
NIE obciąża Uvicorn/Python!
```

```
Request: GET /api/users
    ↓
nginx: "To API, przekazuję do Uvicorn"
    ↓
Uvicorn → FastAPI → Python kod
```

**nginx jest BARDZO szybki w serwowaniu statycznych plików** (100× szybszy niż Python).

---

### 2. HTTPS/SSL termination

```
Klient → HTTPS (encrypted) → nginx
                                ↓
                         nginx dekryptuje
                                ↓
nginx → HTTP (plain) → Uvicorn (lokalnie, bezpieczne)
```

**nginx obsługuje SSL:**
- Certyfikaty (Let's Encrypt)
- Dekrypcja HTTPS
- Uvicorn nie musi się tym martwić!

**Uvicorn otrzymuje plain HTTP** (łatwiejsze, szybsze).

---

### 3. Load Balancing - wiele instancji Uvicorn

```
nginx (port 80)
    ↓ rozdziela requesty:
    ├─ Uvicorn 1 (port 8001)
    ├─ Uvicorn 2 (port 8002)
    ├─ Uvicorn 3 (port 8003)
    └─ Uvicorn 4 (port 8004)
```

nginx równomiernie rozdziela ruch między instancje.

---

### 4. Caching

```
Request 1: GET /api/products
    ↓
nginx → Uvicorn → DB query (100ms)
    ↓
nginx ZAPAMIĘTUJE odpowiedź (cache)

Request 2: GET /api/products (10 sekund później)
    ↓
nginx: "Mam w cache!" → zwraca od razu (1ms)
    ↓
NIE odpytuje Uvicorn!
```

---

### 5. Rate Limiting

```nginx
# nginx.conf
limit_req_zone $binary_remote_addr zone=mylimit:10m rate=10r/s;
# Max 10 requestów/sekundę z jednego IP
```

nginx blokuje nadmiarowe requesty **ZANIM dotrą do Uvicorn**.

---

### 6. Gzip Compression

```
Uvicorn → Response 100KB JSON → nginx
                                   ↓
                            nginx kompresuje (gzip)
                                   ↓
                            10KB (zamiast 100KB!)
                                   ↓
                               Klient
```

Szybszy transfer, mniej bandwidth.

---

### 7. Security - warstwa ochrony

```
Złośliwy request → nginx filtruje → blokuje
Dobry request → nginx przekazuje → Uvicorn
```

**nginx jako firewall:**
- Blokuje złośliwe requesty
- Ukrywa Uvicorn przed światem zewnętrznym
- Uvicorn dostępny tylko z localhost (bezpiecznie!)

---

### 8. Wydajność - nginx jest BARDZO szybki

nginx napisany w C, zoptymalizowany pod HTTP:
- Obsługuje 10,000+ concurrent connections
- Event-driven architecture (nie blokuje się)
- Niskie użycie RAM

**nginx = profesjonalny HTTP server**
**Uvicorn = ASGI interface do Pythona**

Każdy robi to co robi najlepiej!

---

## 4. nginx vs Uvicorn/Gunicorn - różne warstwy!

### KLUCZOWE: To NIE SĄ alternatywy!

```
nginx ≠ Uvicorn
nginx ≠ Gunicorn
```

**To różne warstwy stacku!**

---

### Porównanie ról:

| Aspekt | nginx | Uvicorn/Gunicorn |
|--------|-------|------------------|
| **Typ** | HTTP Server + Reverse Proxy | ASGI/WSGI Server |
| **Rola** | Obsługa HTTP, SSL, static files | Uruchamianie kodu Python |
| **Język** | C (bardzo szybki) | Python + C (szybki) |
| **Wykonuje kod Python?** | ❌ NIE | ✅ TAK |
| **Serwuje static files?** | ✅ TAK (bardzo szybko) | ⚠️ Może, ale wolno |
| **SSL/HTTPS?** | ✅ TAK (native support) | ⚠️ Może, ale niewygodne |
| **Load balancing?** | ✅ TAK | ❌ NIE |
| **Port** | 80 (HTTP), 443 (HTTPS) | 8000 (localhost) |
| **Dostępny z internetu?** | ✅ TAK | ❌ NIE (tylko localhost!) |

---

### Warstwy stacku:

```
┌─────────────────────────────────────────────┐
│  Warstwa 1: HTTP Server / Reverse Proxy    │
│  nginx (port 80/443)                        │
│  - Obsługa HTTP                             │
│  - SSL termination                          │
│  - Static files                             │
│  - Load balancing                           │
└───────────────────┬─────────────────────────┘
                    ↓ przekazuje do
┌─────────────────────────────────────────────┐
│  Warstwa 2: ASGI/WSGI Server                │
│  Uvicorn/Gunicorn (port 8000, localhost)    │
│  - Tłumaczy HTTP → ASGI/WSGI                │
│  - Uruchamia kod Python                     │
└───────────────────┬─────────────────────────┘
                    ↓ wywołuje
┌─────────────────────────────────────────────┐
│  Warstwa 3: Python Application              │
│  FastAPI/Django (Twój kod)                  │
│  - Routing, business logic                  │
└───────────────────┬─────────────────────────┘
                    ↓ zapytania
┌─────────────────────────────────────────────┐
│  Warstwa 4: Database                        │
│  PostgreSQL, MySQL, etc.                    │
└─────────────────────────────────────────────┘
```

**nginx i Uvicorn pracują RAZEM, nie zamiast siebie!**

---

## 5. Konfiguracja nginx jako reverse proxy

### Przykładowa konfiguracja:


In [ ]:
# /etc/nginx/sites-available/myapp

server {
    listen 80;  # Nasłuchuj na porcie 80 (HTTP)
    server_name example.com www.example.com;

    # ========================================================================
    # Statyczne pliki - nginx serwuje SAM (nie przekazuje do Uvicorn)
    # ========================================================================
    location /static/ {
        alias /var/www/myapp/static/;  # Ścieżka do plików na dysku
        expires 30d;  # Cache na 30 dni
    }

    location /media/ {
        alias /var/www/myapp/media/;
        expires 30d;
    }

    # ========================================================================
    # API - przekazuje do Uvicorn (reverse proxy)
    # ========================================================================
    location / {
        proxy_pass http://127.0.0.1:8000;  # Uvicorn na localhost:8000
        
        # Headers - przekazuje informacje o kliencie
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        
        # Timeouts
        proxy_connect_timeout 60s;
        proxy_read_timeout 60s;
    }
}

### Co się dzieje:

```
GET /static/logo.png
    ↓
nginx: location /static/ → serwuje z /var/www/myapp/static/logo.png
    ↓
SZYBKO! (nie angażuje Uvicorn)


GET /api/users
    ↓
nginx: location / → proxy_pass http://127.0.0.1:8000
    ↓
Uvicorn → FastAPI → Python kod → Response
    ↓
nginx → Klient
```

---

### Konfiguracja z HTTPS (Let's Encrypt):


In [ ]:
# Po uruchomieniu: certbot --nginx -d example.com
# Certbot automatycznie doda konfigurację SSL:

server {
    listen 443 ssl http2;  # HTTPS na porcie 443
    server_name example.com;

    # SSL Certificates (dodane przez certbot)
    ssl_certificate /etc/letsencrypt/live/example.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/example.com/privkey.pem;

    # Static files
    location /static/ {
        alias /var/www/myapp/static/;
    }

    # Proxy do Uvicorn
    location / {
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto https;  # HTTPS!
    }
}

# Redirect HTTP → HTTPS
server {
    listen 80;
    server_name example.com;
    return 301 https://$server_name$request_uri;  # Redirect do HTTPS
}

### Flow z HTTPS:

```
Klient → HTTPS request (encrypted)
    ↓
nginx (port 443):
    - Dekryptuje HTTPS
    - Weryfikuje certyfikat
    ↓
nginx → HTTP request (plain, localhost) → Uvicorn (port 8000)
    ↓
Uvicorn → FastAPI → Response
    ↓
nginx:
    - Enkryptuje response (HTTPS)
    ↓
Klient ← HTTPS response (encrypted)
```

**Uvicorn nigdy nie widzi HTTPS - tylko plain HTTP!**

---

## 6. Alternatywy dla nginx

### 1. Apache

**Starszy, bardziej skomplikowany**

```apache
# Apache config
<VirtualHost *:80>
    ServerName example.com
    
    ProxyPass / http://127.0.0.1:8000/
    ProxyPassReverse / http://127.0.0.1:8000/
</VirtualHost>
```

**Pros:**
✅ Dojrzały, stabilny
✅ Wiele modułów

**Cons:**
❌ Wolniejszy niż nginx
❌ Więcej RAM
❌ Bardziej skomplikowana konfiguracja

---

### 2. Caddy

**Nowoczesny, automatyczne HTTPS**

```caddy
# Caddyfile
example.com {
    reverse_proxy localhost:8000
}
```

**Pros:**
✅ **Automatyczne HTTPS** (Let's Encrypt) - zero konfiguracji!
✅ Prostsza konfiguracja niż nginx
✅ Napisany w Go (szybki)

**Cons:**
❌ Mniej dojrzały niż nginx
❌ Mniejsza społeczność

---

### 3. Traefik

**Dla Docker/Kubernetes, auto-discovery**

```yaml
# Traefik automatycznie wykrywa kontenery Docker
labels:
  - "traefik.enable=true"
  - "traefik.http.routers.myapp.rule=Host(`example.com`)"
```

**Pros:**
✅ Automatyczne discovery (Docker labels)
✅ Automatyczne SSL (Let's Encrypt)
✅ Dashboard

**Cons:**
❌ Overkill dla prostych deployments
❌ Wolniejszy niż nginx

---

### 4. HAProxy

**Load balancer głównie**

```haproxy
# HAProxy config
frontend http_front
    bind *:80
    default_backend http_back

backend http_back
    balance roundrobin
    server uvicorn1 127.0.0.1:8001
    server uvicorn2 127.0.0.1:8002
```

**Pros:**
✅ Bardzo wydajny load balancing
✅ Health checks

**Cons:**
❌ Nie serwuje static files (trzeba osobnego serwera)
❌ Bardziej skomplikowany

---

### Porównanie:

| Server | Wydajność | Łatwość | Auto HTTPS | Popularność | Use case |
|--------|-----------|---------|------------|-------------|----------|
| **nginx** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⚠️ Manual | ⭐⭐⭐⭐⭐ | Standard (recommended) |
| **Apache** | ⭐⭐⭐ | ⭐⭐ | ⚠️ Manual | ⭐⭐⭐⭐ | Legacy systems |
| **Caddy** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ✅ Auto | ⭐⭐⭐ | Małe/średnie projekty |
| **Traefik** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ✅ Auto | ⭐⭐⭐ | Docker/Kubernetes |
| **HAProxy** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ❌ Nie | ⭐⭐⭐⭐ | Load balancing głównie |

**Rekomendacja:** nginx dla większości projektów (szybki, sprawdzony, wszechstronny).

---

## 7. Czy można bez nginx?

### Opcja 1: Uvicorn bezpośrednio (bez nginx)

```bash
uvicorn main:app --host 0.0.0.0 --port 80 --workers 4
```

**Kiedy OK:**
✅ Prototypy, małe projekty
✅ Aplikacje bez static files
✅ Development/staging

**Problemy:**
❌ Brak SSL termination (musisz konfigurować w Uvicorn - trudne)
❌ Static files serwowane przez Python (wolne)
❌ Brak load balancing
❌ Brak caching
❌ Mniej security features

---

### Opcja 2: Cloud Load Balancer (AWS ALB, Google Cloud LB)

```
AWS Application Load Balancer
    ↓
Uvicorn (bez nginx)
```

**Kiedy OK:**
✅ Cloud deployment (AWS, GCP, Azure)
✅ Load balancer obsługuje SSL, routing

**Ale:**
⚠️ Nadal potrzebujesz czegoś dla static files (S3, CloudFront)

---

### Rekomendacja:

**Dla większości projektów: UŻYWAJ nginx!**

- Prosty setup
- Darmowy
- Wydajny
- Sprawdzony

**Bez nginx tylko dla:**
- Bardzo małych projektów
- Cloud z managed load balancerami

---

## 🎯 Kluczowe wnioski:

1. **nginx** = HTTP Server + Reverse Proxy (NIE uruchamia Pythona!)
2. **Reverse Proxy** = serwer PRZED aplikacją, przekazuje requesty
3. **nginx ≠ Uvicorn** - to RÓŻNE warstwy stacku (pracują razem!)
4. **Po co nginx:**
   - Static files (szybkie)
   - SSL termination
   - Load balancing
   - Caching, Rate limiting
   - Security
5. **Alternatywy:** Apache (stary), Caddy (prosty), Traefik (Docker)
6. **nginx najpopularniejszy** - szybki, sprawdzony, stabilny

---

## 📚 Dalej:

**Następny notebook:** `05_production_stack.ipynb`
- Pełny production stack (nginx + Uvicorn + FastAPI + PostgreSQL)
- SSL/HTTPS z Let's Encrypt
- Monitoring, logging
- Health checks
- Best practices
- Deployment checklist

---